# Notebook 4 — Modality predictive capacity and learnable-utility diagnostics (CPU)

**Why this notebook exists.** Notebook 3 showed that M4 relies mostly on text (mean gate weight 0.82) and that dropping
audio+video fixes some clips but breaks more. Two questions remain before any new fusion method can be justified:

- **Experiment A — capacity.** Do audio and video, trained *on their own*, carry more sentiment information than M4 extracts
  from them? If dedicated audio/video models clearly beat M4's audio/video-only predictions, that is evidence that M4
  under-uses those modalities (the "modality imbalance" problem studied by OGM-GE and related work). If they do **not**,
  that does **not** show the features are the limit: these baselines are simple pooled MLPs that discard temporal order, so
  a stronger audio/video model could still extract more. That outcome would only say "no evidence of under-use was found
  with pooled baselines".
- **Experiment B — predictability.** Using only information available at prediction time, can we predict for which clips
  audio+video help or hurt, well enough to beat simple fixed rules? This is a frozen, post-hoc version of ideas that already
  exist (true-class-probability-based dynamic fusion, learned confidence, selective prediction). It is **a diagnostic, not a
  proposed method**. A negative result is informative: it would mean per-clip dynamic fusion has little to gain here.

**What is not done:** no existing model is retrained; v1/v2/v3 folders are only read. New outputs go to `mosei_v4_capacity_utility/`.

**Why CPU:** every model here is small (dedicated unimodal MLPs train in well under a minute each in v2's environment),
and the CPU thread count is set to the value recorded by the v2 run, because Notebook 3 showed thread count changes weights.

**Status: exploratory.** The MOSEI test set has already been examined many times (v1, v2, Notebook 3), so nothing found here
counts as confirmed. Any eventual methodological claim needs independent confirmation on another dataset.

**Test labels are never used to train or tune anything.** All choices in Experiment B (regularisation, switching threshold,
best fixed subset) are made on the validation set with video-grouped cross-validation; the test set is used once.

In [1]:
import os, sys, time, json, random, pickle, platform, warnings, hashlib, itertools
IN_COLAB = "google.colab" in sys.modules
TEST_MODE = os.environ.get("EMOFAR_TEST") == "1"
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import sklearn
from scipy import stats
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
DEVICE = "cpu"
torch.backends.cudnn.deterministic = True

Mounted at /content/drive


## 1. Paths and settings (training settings and thread count are taken from the v2 run)

In [2]:
BASE_OUT = "/content/drive/MyDrive/emofar_results"
if TEST_MODE:
    BASE_OUT = os.environ["EMOFAR_OUT"]
V2_DIR = f"{BASE_OUT}/mosei_v2_cpu"                 # read only
V3_DIR = f"{BASE_OUT}/mosei_v3_diagnostics"         # read only (optional, for comparison tables)
OUT_DIR = f"{BASE_OUT}/mosei_v4_capacity_utility"
RESUME = True
N_BOOT = 10000 if not TEST_MODE else 500

for d in [V2_DIR]:
    assert os.path.isdir(d), f"missing folder: {d}"
assert os.path.abspath(OUT_DIR) not in {os.path.abspath(V2_DIR), os.path.abspath(V3_DIR)}
os.makedirs(f"{OUT_DIR}/checkpoints", exist_ok=True)

v2_info = json.load(open(f"{V2_DIR}/run_info.json"))
SEEDS = v2_info["config"]["SEEDS"]
logs = [json.load(open(f"{V2_DIR}/checkpoints/{m}_seed{s}_log.json")) for m in ["M1", "M2", "M3", "M4"] for s in SEEDS]
cfg = logs[0]["train_config"]
assert all(l["train_config"] == cfg for l in logs)
DATA_PATH = cfg["DATA_PATH"] if not TEST_MODE else os.environ["EMOFAR_MOSI"]
D_MODEL, EPOCHS, PATIENCE, BATCH_SIZE, LR, MOD_DROPOUT = (cfg[k] for k in ["D_MODEL", "EPOCHS", "PATIENCE", "BATCH_SIZE", "LR", "MOD_DROPOUT"])
THREADS = int(v2_info["environment"]["torch_threads"])
torch.set_num_threads(THREADS)

ENV = {"python": platform.python_version(), "torch": torch.__version__, "numpy": np.__version__, "pandas": pd.__version__,
       "sklearn": sklearn.__version__, "torch_threads": torch.get_num_threads(), "os_cpu_count": os.cpu_count(),
       "os": platform.platform(), "started_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime())}
try:
    ENV["cpu"] = next(l.split(":", 1)[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name"))
except Exception:
    ENV["cpu"] = platform.processor() or "unknown"
print("This run:", json.dumps(ENV, indent=1))
print("v2 run  :", v2_info["environment"])
print("Training settings (from v2 checkpoints):", cfg, "| seeds", SEEDS, "| threads", THREADS)
json.dump({"environment": ENV, "v2_train_config": cfg, "seeds": SEEDS, "threads": THREADS, "n_boot": N_BOOT},
          open(f"{OUT_DIR}/run_info.json", "w"), indent=2)
RUN_DIR = OUT_DIR                     # the unchanged v2 data cell saves labels/ids into RUN_DIR -> the new folder
CKPT_DIR = f"{V2_DIR}/checkpoints"
LABELS = ["negative", "neutral", "positive"]
MODS = ["text", "audio", "vision"]

This run: {
 "python": "3.13.18",
 "torch": "2.14.1+cpu",
 "numpy": "2.4.6",
 "pandas": "2.2.3",
 "sklearn": "1.6.1",
 "torch_threads": 4,
 "os_cpu_count": 8,
 "os": "Linux-6.6.159+-x86_64-with-glibc2.39",
 "started_utc": "2026-11-24 08:37:51",
 "cpu": "Intel(R) Xeon(R) CPU @ 2.20GHz"
}
v2 run  : {'python': '3.13.18', 'torch': '2.14.1+cpu', 'numpy': '2.4.6', 'pandas': '2.2.3', 'sklearn': '1.6.1', 'cpu': 'Intel(R) Xeon(R) CPU @ 2.20GHz', 'torch_threads': 4, 'os': 'Linux-6.6.159+-x86_64-with-glibc2.39', 'started_utc': '2026-11-24 06:35:51'}
Training settings (from v2 checkpoints): {'D_MODEL': 128, 'EPOCHS': 30, 'PATIENCE': 6, 'BATCH_SIZE': 64, 'LR': 0.0005, 'MOD_DROPOUT': 0.3, 'DATA_PATH': '/content/drive/MyDrive/datasets/CMU-MOSEI/aligned_50.pkl', 'n_train': 16326} | seeds [42, 43, 44] | threads 4


## 2. Load the features (code copied unchanged from v2)

In [3]:
t0 = time.time()
with open(DATA_PATH, "rb") as f:
    raw = pickle.load(f)
print(f"Loaded in {time.time() - t0:.0f}s")

def prep(split):
    d = raw[split]
    mask = d["text_bert"][:, 1, :].astype(bool)                        # (N, 50) real-word steps
    feats = {"text": d["text"].astype(np.float32),
             "audio": np.nan_to_num(d["audio"].astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0),
             "vision": np.nan_to_num(d["vision"].astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)}
    score = d["regression_labels"].astype(np.float32)
    y = np.where(score < 0, 0, np.where(score == 0, 1, 2))
    return {"feats": feats, "mask": mask, "y": y, "score": score, "raw_text": d["raw_text"], "id": d["id"]}

data = {s: prep(s) for s in ["train", "valid", "test"]}
del raw

for m in ["audio", "vision"]:                                           # standardise with train statistics
    tr = data["train"]["feats"][m][data["train"]["mask"]]
    mu, sd = tr.mean(0), tr.std(0) + 1e-6
    for s in data:
        x = (data[s]["feats"][m] - mu) / sd
        data[s]["feats"][m] = np.clip(x, -10, 10) * data[s]["mask"][..., None]

DIMS = {m: data["train"]["feats"][m].shape[-1] for m in MODS}
print("Feature sizes:", DIMS)
for s in data:
    counts = np.bincount(data[s]["y"], minlength=3)
    print(f"{s:5s}: {len(data[s]['y']):6,} clips | " + " / ".join(f"{LABELS[i]} {counts[i]:,}" for i in range(3)))
np.save(f"{RUN_DIR}/test_labels.npy", data["test"]["y"]); np.save(f"{RUN_DIR}/valid_labels.npy", data["valid"]["y"])
np.save(f"{RUN_DIR}/test_scores.npy", data["test"]["score"])
pd.Series(data["test"]["id"]).to_csv(f"{RUN_DIR}/test_ids.csv", index=False, header=["id"])
# Exact agreement with the v2 artifacts: same test clips in the same order, same labels
v2_test_ids = pd.read_csv(f"{V2_DIR}/test_ids.csv", dtype=str)["id"].values
assert len(v2_test_ids) == len(data["test"]["id"]) and (v2_test_ids == np.asarray(data["test"]["id"]).astype(str)).all(), \
    "test clip IDs or their order differ from the v2 run"
assert (np.load(f"{V2_DIR}/test_labels.npy") == data["test"]["y"]).all(), "test labels differ from the v2 run"
assert (np.load(f"{V2_DIR}/valid_labels.npy") == data["valid"]["y"]).all(), "validation labels differ from the v2 run"
assert len(data["train"]["y"]) == cfg["n_train"], "training set size differs from the v2 run"
print("Test IDs (order), test labels and validation labels match the v2 artifacts exactly.")
np.save(f"{OUT_DIR}/valid_ids.npy", np.asarray(data["valid"]["id"]).astype(str))

Loaded in 40s
Feature sizes: {'text': 768, 'audio': 74, 'vision': 35}
train: 16,326 clips | negative 4,738 / neutral 3,540 / positive 8,048
valid:  1,871 clips | negative 506 / neutral 433 / positive 932
test :  4,659 clips | negative 1,350 / neutral 1,025 / positive 2,284
Test IDs (order), test labels and validation labels match the v2 artifacts exactly.


## 3. Model definitions (copied unchanged from v2)

In [4]:
def masked_mean(x, mask):
    m = mask.unsqueeze(-1).float()
    return (x * m).sum(1) / m.sum(1).clamp(min=1)

class TextOnly(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(DIMS["text"], D_MODEL), nn.ReLU(), nn.Dropout(0.2), nn.Linear(D_MODEL, 3))
    def forward(self, x, mask, avail, return_gate=False):
        out = self.net(masked_mean(x["text"], mask))
        return (out, None) if return_gate else out

class PaperRMSR(nn.Module):
    """Equation-level adaptation of EMO-FAR's RMSR (Eq. 10-14)."""
    def __init__(self):
        super().__init__()
        self.proj = nn.ModuleDict({m: nn.Linear(DIMS[m], D_MODEL) for m in MODS})          # Eq. 10
        self.u = nn.ParameterDict({m: nn.Parameter(torch.randn(D_MODEL) * 0.05) for m in MODS})  # Eq. 11
        self.Wq, self.Wk, self.Wv = (nn.Linear(D_MODEL, D_MODEL) for _ in range(3))
        self.cls = nn.Sequential(nn.Dropout(0.2), nn.Linear(D_MODEL, 3))
    def forward(self, x, mask, avail, return_gate=False):
        z = {m: self.proj[m](masked_mean(x[m], mask)) for m in MODS}
        q = torch.stack([torch.sigmoid((z[m] * self.u[m]).sum(-1)) for m in MODS], -1)   # Eq. 11
        gamma = q.softmax(-1)                                                             # Eq. 12
        att = {}
        for m in ["audio", "vision"]:                                                     # Eq. 13
            score = (self.Wq(z["text"]) * self.Wk(z[m])).sum(-1, keepdim=True) / D_MODEL ** 0.5
            att[m] = score.softmax(-1) * self.Wv(z[m])
        h = gamma[:, :1] * z["text"] + gamma[:, 1:2] * att["audio"] + gamma[:, 2:] * att["vision"]   # Eq. 14
        out = self.cls(h)
        return (out, gamma) if return_gate else out

class FixedFusion(nn.Module):
    """Token-level cross-attention + masked gating."""
    def __init__(self):
        super().__init__()
        self.proj = nn.ModuleDict({m: nn.Sequential(nn.Linear(DIMS[m], D_MODEL), nn.LayerNorm(D_MODEL)) for m in MODS})
        self.xattn = nn.ModuleDict({m: nn.MultiheadAttention(D_MODEL, 4, batch_first=True, dropout=0.1)
                                    for m in ["audio", "vision"]})
        self.gate = nn.Linear(3 * D_MODEL, 3)
        self.cls = nn.Sequential(nn.Dropout(0.2), nn.Linear(D_MODEL, D_MODEL), nn.ReLU(), nn.Linear(D_MODEL, 3))
    def forward(self, x, mask, avail, return_gate=False):
        p = {m: self.proj[m](x[m]) * avail[:, i, None, None] for i, m in enumerate(MODS)}
        h = {"text": masked_mean(p["text"], mask)}
        for m in ["audio", "vision"]:
            attended, _ = self.xattn[m](p["text"], p[m], p[m], key_padding_mask=~mask)
            h[m] = masked_mean(attended + p[m], mask)
        logits = self.gate(torch.cat([h[m] for m in MODS], -1))
        logits = logits.masked_fill(avail == 0, float("-inf"))
        w = logits.softmax(-1)
        fused = sum(w[:, i:i + 1] * h[m] for i, m in enumerate(MODS))
        out = self.cls(fused)
        return (out, w) if return_gate else out

MODELS = {"M1 Text-only": (TextOnly, 0.0),
          "M2 EMO-FAR RMSR (as written)": (PaperRMSR, 0.0),
          "M3 Fixed fusion": (FixedFusion, 0.0),
          "M4 Fixed fusion + modality dropout": (FixedFusion, MOD_DROPOUT)}
SHORT = {name: name.split()[0] for name in MODELS}                     # "M1" ... "M4"

## 4. Training/evaluation functions (copied unchanged from v2)

In [5]:
from sklearn.metrics import f1_score, accuracy_score

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def sample_avail(n, rates, rng):
    """rates = (p_text, p_audio, p_vision): chance each modality is hidden. At least one stays."""
    avail = (rng.rand(n, 3) >= np.array(rates)).astype(np.float32)
    none_left = avail.sum(1) == 0
    avail[none_left, rng.randint(0, 3, none_left.sum())] = 1
    return avail

def to_batch(d, idx, avail):
    x = {m: torch.from_numpy(d["feats"][m][idx]).to(DEVICE) for m in MODS}
    for i, m in enumerate(MODS):
        x[m] = x[m] * torch.from_numpy(avail[:, i]).to(DEVICE)[:, None, None]
    return x, torch.from_numpy(d["mask"][idx]).to(DEVICE), torch.from_numpy(avail).to(DEVICE)

@torch.no_grad()
def predict(model, d, rates=(0, 0, 0), seed=0):
    model.eval(); rng = np.random.RandomState(seed); out = []
    for i in range(0, len(d["y"]), 256):
        idx = np.arange(i, min(i + 256, len(d["y"])))
        x, mask, avail = to_batch(d, idx, sample_avail(len(idx), rates, rng))
        out.append(model(x, mask, avail).softmax(-1).cpu().numpy())
    return np.concatenate(out)

def ece_score(probs, y, n_bins=15):
    conf, pred = probs.max(1), probs.argmax(1); ece = 0.0
    for lo, hi in zip(np.linspace(0, 1, n_bins + 1)[:-1], np.linspace(0, 1, n_bins + 1)[1:]):
        sel = (conf > lo) & (conf <= hi)
        if sel.any():
            ece += sel.mean() * abs((pred[sel] == y[sel]).mean() - conf[sel].mean())
    return float(ece)

def evaluate(probs, d):
    y, pred = d["y"], probs.argmax(1)
    nz = d["score"] != 0
    acc2 = ((probs[nz, 2] > probs[nz, 0]) == (d["score"][nz] > 0)).mean()
    return {"accuracy": accuracy_score(y, pred), "macro_f1": f1_score(y, pred, average="macro"),
            "acc2": float(acc2), "ece": ece_score(probs, y)}

def train(name, seed):
    cls, drop = MODELS[name]
    set_seed(seed)
    model = cls().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    counts = np.bincount(data["train"]["y"], minlength=3)
    class_w = torch.tensor(counts.sum() / (3 * counts), dtype=torch.float, device=DEVICE)
    rng = np.random.RandomState(seed); tr = data["train"]
    best, best_state, bad = -1, None, 0
    for ep in range(EPOCHS):
        model.train()
        order = rng.permutation(len(tr["y"]))
        for i in range(0, len(order), BATCH_SIZE):
            idx = order[i:i + BATCH_SIZE]
            x, mask, avail = to_batch(tr, idx, sample_avail(len(idx), (drop, drop, drop), rng))
            loss = F.cross_entropy(model(x, mask, avail), torch.from_numpy(tr["y"][idx]).to(DEVICE), weight=class_w)
            opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        val = evaluate(predict(model, data["valid"]), data["valid"])["macro_f1"]
        if val > best:
            best, bad = val, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    model.load_state_dict(best_state)
    print(f"  {name:38s} seed {seed}: best valid macro-F1 = {best:.4f}  (stopped after epoch {ep + 1})")
    return model, float(best), ep + 1

def save_atomic(save_fn, path):
    """Write to a temporary file first, then rename: a disconnect never leaves a half-written file."""
    tmp = path + ".tmp"
    save_fn(tmp)
    os.replace(tmp, path)

## 5. Shared helpers: video groups and a video-grouped paired bootstrap
A MOSEI clip id looks like `videoID$_$clipNumber`. Clips of the same video are resampled together.

In [6]:
def video_of(ids):
    return np.array([str(i).split("$_$")[0] for i in ids])

def f1_from_counts(cm):
    cm = cm.reshape(cm.shape[:-1] + (3, 3)); tp = np.diagonal(cm, axis1=-2, axis2=-1)
    fp = cm.sum(-2) - tp; fn = cm.sum(-1) - tp; den = 2 * tp + fp + fn
    return np.where(den > 0, 2 * tp / np.maximum(den, 1e-12), 0).mean(-1)

y_test = data["test"]["y"]; y_val = data["valid"]["y"]
vid_test = video_of(data["test"]["id"]); _, vidx_test = np.unique(vid_test, return_inverse=True); G_TEST = vidx_test.max() + 1
vid_val = video_of(data["valid"]["id"])
_rng = np.random.RandomState(2026)
W_TEST = np.stack([np.bincount(_rng.randint(0, G_TEST, G_TEST), minlength=G_TEST) for _ in range(N_BOOT)]).astype(np.float32)

def video_counts(pred):
    c = np.zeros((G_TEST, 9)); np.add.at(c, (vidx_test, y_test * 3 + pred), 1); return c

SEED_DRAWS = np.random.RandomState(99).randint(0, len(SEEDS), (N_BOOT, len(SEEDS)))   # for the seed+video bootstrap

def _p(b):
    return min(1.0, 2 * (min((b <= 0).sum(), (b >= 0).sum()) + 1) / (len(b) + 1))

def grouped_boot(preds_a, preds_b):
    """preds_a / preds_b: one prediction vector per training seed (same order).
    Returns a dict with:
      per-seed differences and their SD (training-seed variation, observed);
      video bootstrap  - resamples test videos, training seeds fixed (test-sample uncertainty only);
      seed+video boot. - also resamples the training seeds with replacement (adds training-seed uncertainty).
    With 3 seeds the seed resampling has only 10 distinct outcomes, so it is a rough sensitivity check."""
    per_seed = np.array([f1_score(y_test, a, average="macro") - f1_score(y_test, b, average="macro") for a, b in zip(preds_a, preds_b)])
    d = np.stack([f1_from_counts(W_TEST @ video_counts(a)) - f1_from_counts(W_TEST @ video_counts(b))
                  for a, b in zip(preds_a, preds_b)])                     # (seeds, N_BOOT)
    video = d.mean(0)
    seed_video = d[SEED_DRAWS, np.arange(N_BOOT)[:, None]].mean(1)
    ci = lambda b: f"[{100*np.percentile(b, 2.5):+.2f}, {100*np.percentile(b, 97.5):+.2f}]"
    return {"mean_diff_pp": 100 * per_seed.mean(), "per_seed_pp": ", ".join(f"{100*x:+.2f}" for x in per_seed),
            "sd_over_seeds_pp": 100 * per_seed.std(ddof=1) if len(per_seed) > 1 else np.nan,
            "seeds_positive": f"{int((per_seed > 0).sum())}/{len(per_seed)}",
            "video_CI95_pp": ci(video), "video_p": _p(video), "seed+video_CI95_pp": ci(seed_video), "seed+video_p": _p(seed_video)}

def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); out = np.empty_like(p); run = 0.0
    for r, i in enumerate(o):
        run = max(run, min(1.0, (len(p) - r) * p[i])); out[i] = run
    return out

print(f"test: {len(y_test)} clips in {G_TEST} videos | valid: {len(y_val)} clips in {len(set(vid_val))} videos")

test: 4659 clips in 676 videos | valid: 1871 clips in 300 videos


## 6. Experiment A — dedicated audio-only, video-only and audio+video models

Model: the **same architecture as M1** (masked mean over time → Linear(·,128) → ReLU → Dropout 0.2 → Linear(128,3)),
with the chosen modality's features as input (audio+video: the two pooled vectors concatenated). Same training function,
epochs, early stopping, class weights, learning rate and seeds as v2. No modality dropout.

**Control (run first):** the same class with *text* input must reproduce the saved M1 checkpoint **bit for bit** (weight hash).
If it does not, this environment does not match v2 and Experiment A's numbers would not be comparable — the cell stops.

Limitation: mean pooling discards temporal order. These baselines measure what *pooled* audio/video features carry,
which is also what M4 effectively sees when text is hidden.

In [7]:
class PooledMLP(nn.Module):
    def __init__(self, mods):
        super().__init__()
        self.mods = list(mods)
        in_dim = sum(DIMS[m] for m in self.mods)
        self.net = nn.Sequential(nn.Linear(in_dim, D_MODEL), nn.ReLU(), nn.Dropout(0.2), nn.Linear(D_MODEL, 3))
    def forward(self, x, mask, avail, return_gate=False):
        out = self.net(torch.cat([masked_mean(x[m], mask) for m in self.mods], -1))
        return (out, None) if return_gate else out

NEW_MODELS = {"U0 Text-only control": ["text"], "U1 Audio-only": ["audio"], "U2 Video-only": ["vision"], "U3 Audio+video": ["audio", "vision"]}
for name, mods in NEW_MODELS.items():
    MODELS[name] = ((lambda mods: lambda: PooledMLP(mods))(mods), 0.0)
TRAIN_CONFIG = dict(cfg, threads=THREADS)

def weight_hash(state):
    return hashlib.md5(b"".join(v.numpy().tobytes() for v in state.values())).hexdigest()[:12]

def train_or_load(name, seed):
    tag = f"{name.split()[0]}_seed{seed}"
    ckpt, logf = f"{OUT_DIR}/checkpoints/{tag}.pt", f"{OUT_DIR}/checkpoints/{tag}_log.json"
    if RESUME and os.path.exists(ckpt) and os.path.exists(logf):
        log = json.load(open(logf))
        assert log["train_config"] == TRAIN_CONFIG, f"{ckpt} was trained with other settings; use a new OUT_DIR"
        model = MODELS[name][0](); model.load_state_dict(torch.load(ckpt, map_location="cpu"))
        print(f"  {name:38s} seed {seed}: loaded saved checkpoint")
    else:
        t1 = time.time(); model, best, ep = train(name, seed)
        log = {"model": name, "seed": seed, "best_valid_macro_f1": best, "stopped_epoch": ep,
               "train_seconds": round(time.time() - t1, 1), "train_config": TRAIN_CONFIG}
        save_atomic(lambda p: torch.save(model.state_dict(), p), ckpt)
        save_atomic(lambda p: json.dump(log, open(p, "w"), indent=2), logf)
    model.eval()
    for split in ["valid", "test"]:
        np.save(f"{OUT_DIR}/checkpoints/{tag}_{split}_probs.npy", predict(model, data[split]))
    return model, log

# Control: must reproduce the v2 M1 checkpoint exactly
ctrl, _ = train_or_load("U0 Text-only control", SEEDS[0])
h_ctrl = weight_hash(ctrl.state_dict())
h_m1 = weight_hash(torch.load(f"{CKPT_DIR}/M1_seed{SEEDS[0]}.pt", map_location="cpu"))
print(f"Control hash {h_ctrl} vs v2 M1 hash {h_m1}:", "IDENTICAL" if h_ctrl == h_m1 else "DIFFERENT")
json.dump({"control_hash": h_ctrl, "v2_m1_hash": h_m1, "identical": h_ctrl == h_m1},
          open(f"{OUT_DIR}/control_check.json", "w"), indent=2)
assert h_ctrl == h_m1, "Environment does not reproduce v2 training. Stop: Experiment A would not be comparable."

  U0 Text-only control                   seed 42: best valid macro-F1 = 0.6244  (stopped after epoch 12)
Control hash dd8c7e776029 vs v2 M1 hash dd8c7e776029: IDENTICAL


In [8]:
rows, unimodal_test = [], {}
for name in ["U1 Audio-only", "U2 Video-only", "U3 Audio+video"]:
    for seed in SEEDS:
        model, log = train_or_load(name, seed)
        p = np.load(f"{OUT_DIR}/checkpoints/{name.split()[0]}_seed{seed}_test_probs.npy")
        unimodal_test[(name, seed)] = p
        r = evaluate(p, data["test"]); r.update(model=name, seed=seed, best_valid_macro_f1=log["best_valid_macro_f1"],
                                              stopped_epoch=log["stopped_epoch"]); rows.append(r)
uni = pd.DataFrame(rows); uni.to_csv(f"{OUT_DIR}/expA_unimodal_per_seed.csv", index=False)
print(uni.pivot(index="seed", columns="model", values="macro_f1").round(4))
uni.groupby("model")[["accuracy", "macro_f1", "acc2", "ece"]].agg(["mean", "std"]).round(4)

  U1 Audio-only                          seed 42: best valid macro-F1 = 0.4192  (stopped after epoch 18)
  U1 Audio-only                          seed 43: best valid macro-F1 = 0.4236  (stopped after epoch 22)
  U1 Audio-only                          seed 44: best valid macro-F1 = 0.4127  (stopped after epoch 21)
  U2 Video-only                          seed 42: best valid macro-F1 = 0.4268  (stopped after epoch 16)
  U2 Video-only                          seed 43: best valid macro-F1 = 0.4214  (stopped after epoch 15)
  U2 Video-only                          seed 44: best valid macro-F1 = 0.4174  (stopped after epoch 13)
  U3 Audio+video                         seed 42: best valid macro-F1 = 0.4449  (stopped after epoch 23)
  U3 Audio+video                         seed 43: best valid macro-F1 = 0.4468  (stopped after epoch 23)
  U3 Audio+video                         seed 44: best valid macro-F1 = 0.4397  (stopped after epoch 22)
model  U1 Audio-only  U2 Video-only  U3 Audio+video
see

accuracy         macro_f1            acc2             ece  \
                   mean     std     mean     std    mean     std    mean   
model                                                                      
U1 Audio-only    0.4522  0.0091   0.4106  0.0010  0.6325  0.0097  0.0189   
U2 Video-only    0.4469  0.0065   0.4186  0.0078  0.6192  0.0051  0.0163   
U3 Audio+video   0.4725  0.0029   0.4374  0.0042  0.6480  0.0027  0.0446   

                        
                   std  
model                   
U1 Audio-only   0.0029  
U2 Video-only   0.0058  
U3 Audio+video  0.0007

### 6b. Dedicated models vs. what M4 achieves with the same modalities (test set)
M4's numbers are M4 evaluated with only these modalities available (recomputed here from the v2 checkpoints).
Difference = dedicated − M4 (seeds averaged), with a video-grouped 95 % bootstrap interval.
Reading: a clearly positive difference is evidence that M4 extracts less from audio/video than a model trained on them alone.
A difference near zero or negative does **not** show that the features are the limit (pooled baselines are weak; see above).

In [9]:
@torch.no_grad()
def subset_probs(model, d, avail_row):
    model.eval(); out, gates = [], []
    n = len(d["y"])
    for i in range(0, n, 256):
        idx = np.arange(i, min(i + 256, n))
        avail = np.tile(np.asarray(avail_row, dtype=np.float32), (len(idx), 1))
        x, mask, av = to_batch(d, idx, avail)
        logits, g = model(x, mask, av, return_gate=True)
        out.append(logits.softmax(-1).numpy())
        if g is not None: gates.append(g.numpy())
    return np.concatenate(out), (np.concatenate(gates) if gates else None)

M4_NAME = "M4 Fixed fusion + modality dropout"
m4 = {}
for seed in SEEDS:
    m = MODELS[M4_NAME][0](); m.load_state_dict(torch.load(f"{CKPT_DIR}/M4_seed{seed}.pt", map_location="cpu")); m.eval()
    m4[seed] = m
    saved = np.load(f"{CKPT_DIR}/M4_seed{seed}_test_probs.npy")
    assert (predict(m, data["test"]).argmax(1) == saved.argmax(1)).all(), "M4 checkpoint does not reproduce v2 predictions"

SUBS = {"T": (1, 0, 0), "A": (0, 1, 0), "V": (0, 0, 1), "A+V": (0, 1, 1), "T+A": (1, 1, 0), "T+V": (1, 0, 1), "T+A+V": (1, 1, 1)}
sp, pass_seconds = {}, []
os.makedirs(f"{OUT_DIR}/m4_subset_probs", exist_ok=True)
for split in ["valid", "test"]:
    for s in SEEDS:
        for k, row in SUBS.items():
            t1 = time.perf_counter()
            sp[(split, s, k)] = subset_probs(m4[s], data[split], row)
            if split == "test": pass_seconds.append(time.perf_counter() - t1)
            np.save(f"{OUT_DIR}/m4_subset_probs/M4_seed{s}_{split}_{k.replace('+', '')}.npy", sp[(split, s, k)][0])
SEC_PER_PASS = float(np.median(pass_seconds))      # one M4 forward pass over the whole test set, this machine
print(f"One M4 forward pass over the {len(y_test)} test clips: median {SEC_PER_PASS:.2f} s ({1000*SEC_PER_PASS/len(y_test):.3f} ms/clip)")

rows = []
for name, k in [("U1 Audio-only", "A"), ("U2 Video-only", "V"), ("U3 Audio+video", "A+V")]:
    ded = [unimodal_test[(name, s)].argmax(1) for s in SEEDS]
    sub = [sp[("test", s, k)][0].argmax(1) for s in SEEDS]
    rows.append({"modalities": k, "dedicated_macro_f1": np.mean([f1_score(y_test, a, average="macro") for a in ded]),
                 "M4_subset_macro_f1": np.mean([f1_score(y_test, b, average="macro") for b in sub]), **grouped_boot(ded, sub)})
capA = pd.DataFrame(rows)
capA["video_p_holm"] = holm(capA["video_p"]); capA["seed+video_p_holm"] = holm(capA["seed+video_p"])
capA.to_csv(f"{OUT_DIR}/expA_dedicated_vs_M4_subset.csv", index=False)
capA.round(4)

One M4 forward pass over the 4659 test clips: median 1.36 s (0.292 ms/clip)


,modalities,dedicated_macro_f1,M4_subset_macro_f1,mean_diff_pp,per_seed_pp,sd_over_seeds_pp,seeds_positive,video_CI95_pp,video_p,seed+video_CI95_pp,seed+video_p,video_p_holm,seed+video_p_holm
0,A,0.4106,0.4002,1.0444,"+1.18, +1.59, +0.37",0.6205,3/3,"[-0.02, +2.15]",0.0578,"[-0.40, +2.46]",0.1420,0.0578,0.1420
1,V,0.4186,0.3945,2.4058,"+2.90, +3.00, +1.32",0.9451,3/3,"[+1.20, +3.57]",0.0002,"[+0.74, +4.05]",0.0070,0.0006,0.0210
2,A+V,0.4374,0.4176,1.9832,"+1.60, +2.40, +1.95",0.4001,3/3,"[+0.68, +3.34]",0.0022,"[+0.44, +3.63]",0.0118,0.0044,0.0236


## 7. Experiment B — can per-clip usefulness of audio+video be predicted at prediction time?

**Decision studied:** for each clip, use M4's normal prediction (T+A+V) or M4's text-only prediction (T).

**Target (used only for fitting on validation):** Δ = log p(true label | T+A+V) − log p(true label | T).
Δ < 0 means audio+video lowered the probability of the correct answer on that clip.

**Features (all available at prediction time, from frozen M4):** class probabilities, maximum probability and entropy under
T, A, V, A+V and T+A+V; the three gate weights under T+A+V; whether the T and T+A+V predictions agree; whether the A+V and T
predictions agree; log utterance length. Nothing uses the label.

**Selectors compared on the test set (all tuned on validation only):**
| Selector | Rule |
|---|---|
| S0 M4 (T+A+V) | always the normal prediction (reference) |
| S1 Text only | always T |
| S2 Best fixed subset | the one of T, T+A, T+V, T+A+V with the best validation macro-F1, used for every clip |
| S3 Confidence rule | use T when its maximum probability is higher than T+A+V's |
| S4 Probability average | average of the T and T+A+V probability vectors |
| S5 Learned selector | ridge regression predicts Δ; switch to T for the clips with the lowest predicted Δ. The amount of regularisation and the share of clips switched (0 %, 1 %, 2 %, 5 %, 10 %, 20 %, 30 %) are chosen by 5-fold **video-grouped** cross-validation on the validation set. 0 % (never switch) is allowed, so S5 can fall back to S0. |
| R1 M1 text-only model | the saved predictions of the **independently trained** text-only model M1 (v2). Not a selector — a reference. |

**Comparisons reported:** every selector and R1 against S0, and S5 directly against S2, S3, S4 and R1 (10 comparisons,
Holm-corrected together). Each with per-seed differences, a video-grouped bootstrap (training seeds fixed) and a
seed+video bootstrap (training seeds also resampled).

**Inference cost:** S0, S1, S2 need one M4 forward pass; S3 and S4 need two (T and T+A+V); S5 needs five
(T, A, V, A+V, T+A+V) plus a ridge prediction; R1 needs one M1 pass (much smaller model). Measured times are reported.

**Known weaknesses, stated in advance:** (1) the validation set (~1,871 clips) was also used for early stopping of M4, so M4's
validation predictions are slightly optimistic; (2) clips where switching would fix an error are rare (Notebook 3: on average
93 of 4,659 test clips for M4), so the learned selector has few useful examples; (3) results are for M4 only.

In [10]:
def feats(split, s):
    P = {k: sp[(split, s, k)][0] for k in SUBS}
    gate = sp[(split, s, "T+A+V")][1]
    cols = []
    for k in ["T", "A", "V", "A+V", "T+A+V"]:
        p = np.clip(P[k], 1e-12, 1)
        cols += [p, p.max(1, keepdims=True), -(p * np.log(p)).sum(1, keepdims=True)]
    cols += [gate, (P["T"].argmax(1) == P["T+A+V"].argmax(1))[:, None].astype(float),
             (P["A+V"].argmax(1) == P["T"].argmax(1))[:, None].astype(float),
             np.log(data[split]["mask"].sum(1, keepdims=True).clip(min=1)).astype(float)]
    return np.hstack(cols)

def delta_target(split, s):
    y = data[split]["y"]; n = len(y)
    pa, pt = sp[(split, s, "T+A+V")][0], sp[(split, s, "T")][0]
    return np.log(np.clip(pa[np.arange(n), y], 1e-12, 1)) - np.log(np.clip(pt[np.arange(n), y], 1e-12, 1))

ALPHAS = [0.1, 1.0, 10.0, 100.0, 1000.0]
FRACS = [0.0, 0.01, 0.02, 0.05, 0.10, 0.20, 0.30]

def switch(pred_delta, thr, p_tav, p_t):
    use_t = pred_delta < thr
    return np.where(use_t[:, None], p_t, p_tav), use_t

def fit_selector(s):
    X, yd = feats("valid", s), delta_target("valid", s)
    p_tav, p_t = sp[("valid", s, "T+A+V")][0], sp[("valid", s, "T")][0]
    gkf = GroupKFold(n_splits=5)
    best = None; cv_rows = []
    for a in ALPHAS:
        oof = {f: np.zeros((len(yd), 3)) for f in FRACS}
        for tr, te in gkf.split(X, yd, groups=vid_val):
            reg = make_pipeline(StandardScaler(), Ridge(alpha=a)).fit(X[tr], yd[tr])
            pr_tr, pr_te = reg.predict(X[tr]), reg.predict(X[te])
            for f in FRACS:
                thr = -np.inf if f == 0 else np.quantile(pr_tr, f)
                oof[f][te] = switch(pr_te, thr, p_tav[te], p_t[te])[0]
        for f in FRACS:
            score = f1_score(y_val, oof[f].argmax(1), average="macro")
            cv_rows.append({"seed": s, "alpha": a, "switch_fraction": f, "oof_valid_macro_f1": score})
            if best is None or score > best[0] + 1e-12:
                best = (score, a, f)
    _, a, f = best
    reg = make_pipeline(StandardScaler(), Ridge(alpha=a)).fit(X, yd)
    thr = -np.inf if f == 0 else np.quantile(reg.predict(X), f)
    return reg, thr, a, f, pd.DataFrame(cv_rows)

In [11]:
SELECTORS = ["S0 M4 (T+A+V)", "S1 Text only", "S2 Best fixed subset", "S3 Confidence rule", "S4 Probability average",
             "S5 Learned selector", "R1 M1 text-only model"]
SEL_TAG = {k: k.split()[0] for k in SELECTORS}
os.makedirs(f"{OUT_DIR}/selector_outputs", exist_ok=True)
test_pred = {k: [] for k in SELECTORS}
ridge_seconds = []
rows, diag_rows, cv_all = [], [], []
for s in SEEDS:
    P = {k: sp[("test", s, k)][0] for k in SUBS}
    Pv = {k: sp[("valid", s, k)][0] for k in SUBS}
    best_fixed = max(["T", "T+A", "T+V", "T+A+V"], key=lambda k: f1_score(y_val, Pv[k].argmax(1), average="macro"))
    reg, thr, a, f, cv = fit_selector(s); cv_all.append(cv)
    t1 = time.perf_counter(); X_test = feats("test", s); pred_delta = reg.predict(X_test); ridge_seconds.append(time.perf_counter() - t1)
    p_s5, used_t = switch(pred_delta, thr, P["T+A+V"], P["T"])
    np.save(f"{OUT_DIR}/selector_outputs/S5_seed{s}_test_predicted_delta.npy", pred_delta)
    np.save(f"{OUT_DIR}/selector_outputs/S5_seed{s}_test_switched_to_T.npy", used_t)
    outs = {"S0 M4 (T+A+V)": P["T+A+V"], "S1 Text only": P["T"], "S2 Best fixed subset": P[best_fixed],
            "S3 Confidence rule": np.where((P["T"].max(1) > P["T+A+V"].max(1))[:, None], P["T"], P["T+A+V"]),
            "S4 Probability average": (P["T"] + P["T+A+V"]) / 2, "S5 Learned selector": p_s5,
            "R1 M1 text-only model": np.load(f"{CKPT_DIR}/M1_seed{s}_test_probs.npy")}
    for k, p in outs.items():
        test_pred[k].append(p.argmax(1))
        np.save(f"{OUT_DIR}/selector_outputs/{SEL_TAG[k]}_seed{s}_test_probs.npy", p.astype(np.float32))
        r = evaluate(p, data["test"]); r.update(seed=s, selector=k); rows.append(r)
    # diagnostics of the learned Δ predictor on the test set (labels used only for scoring here)
    true_delta = delta_target("test", s)
    ok_tav, ok_t = P["T+A+V"].argmax(1) == y_test, P["T"].argmax(1) == y_test
    fixed, broken = ~ok_tav & ok_t, ok_tav & ~ok_t
    diag_rows.append({"seed": s, "chosen_alpha": a, "chosen_switch_fraction": f, "best_fixed_subset": best_fixed,
                      "spearman_pred_vs_true_delta": stats.spearmanr(pred_delta, true_delta)[0],
                      "AUROC_detect_AV_hurts(true_delta<0)": roc_auc_score(true_delta < 0, -pred_delta),
                      "AUROC_detect_fixable_clips": roc_auc_score(fixed, -pred_delta) if fixed.any() else np.nan,
                      "AUROC_confidence_rule_fixable": roc_auc_score(fixed, P["T"].max(1) - P["T+A+V"].max(1)) if fixed.any() else np.nan,
                      "clips_switched": int(used_t.sum()), "switched_and_fixed": int((used_t & fixed).sum()),
                      "switched_and_broken": int((used_t & broken).sum()),
                      "fixable_clips_total": int(fixed.sum()), "breakable_clips_total": int(broken.sum()),
                      "accuracy_ceiling_gain_pp": 100 * fixed.mean()})
sel = pd.DataFrame(rows); sel.to_csv(f"{OUT_DIR}/expB_selectors_per_seed.csv", index=False)
diag = pd.DataFrame(diag_rows); diag.to_csv(f"{OUT_DIR}/expB_predictor_diagnostics.csv", index=False)
pd.concat(cv_all).to_csv(f"{OUT_DIR}/expB_validation_cv.csv", index=False)
print("Test macro-F1 per seed:")
print(sel.pivot(index="seed", columns="selector", values="macro_f1")[SELECTORS].round(4).to_string())
sel.groupby("selector", sort=False)[["accuracy", "macro_f1", "acc2", "ece"]].agg(["mean", "std"]).round(4)

Test macro-F1 per seed:
selector  S0 M4 (T+A+V)  S1 Text only  S2 Best fixed subset  S3 Confidence rule  S4 Probability average  S5 Learned selector  R1 M1 text-only model
seed                                                                                                                                               
42               0.6373        0.6256                0.6256              0.6306                  0.6315               0.6353                 0.6289
43               0.6287        0.6236                0.6287              0.6280                  0.6293               0.6327                 0.6361
44               0.6374        0.6309                0.6315              0.6347                  0.6350               0.6375                 0.6308


accuracy         macro_f1            acc2          \
                           mean     std     mean     std    mean     std   
selector                                                                   
S0 M4 (T+A+V)            0.6619  0.0082   0.6345  0.0050  0.8369  0.0048   
S1 Text only             0.6478  0.0067   0.6267  0.0038  0.8357  0.0036   
S2 Best fixed subset     0.6552  0.0083   0.6286  0.0030  0.8359  0.0041   
S3 Confidence rule       0.6564  0.0074   0.6311  0.0034  0.8364  0.0045   
S4 Probability average   0.6571  0.0069   0.6319  0.0029  0.8378  0.0033   
S5 Learned selector      0.6634  0.0066   0.6352  0.0024  0.8384  0.0044   
R1 M1 text-only model    0.6525  0.0088   0.6319  0.0037  0.8385  0.0046   

                           ece          
                          mean     std  
selector                                
S0 M4 (T+A+V)           0.0206  0.0057  
S1 Text only            0.0404  0.0085  
S2 Best fixed subset    0.0320  0.0126  
S3 Confidence rule      0.0453  0.0123  
S4 Probability average  0.0248  0.0053  
S5 Learned selector     0.0208  0.0066  
R1 M1 text-only model   0.0177  0.0048

### 7b. Learned-predictor diagnostics and statistical comparison with M4
- `spearman_pred_vs_true_delta`: does the predicted Δ rank test clips like the true Δ? (0 = no information)
- `AUROC_*`: 0.5 = chance; 1.0 = perfect detection of clips where audio+video hurt / where switching would fix an error.
  The confidence-rule AUROC is a simple baseline for the same detection task.
- `accuracy_ceiling_gain_pp`: the largest possible accuracy gain of **any** T-vs-T+A+V switching rule (switching exactly the
  fixable clips). Unlike Notebook 3's label-based "oracle" macro-F1, this is an exact upper bound — for accuracy only.
- The comparison table (next cell): 10 comparisons, macro-F1 difference in percentage points, Holm-corrected together
  separately for the video bootstrap and for the seed+video bootstrap.

In [12]:
print(diag.round(4).to_string(index=False))
PAIRS_B = [(k, "S0 M4 (T+A+V)") for k in SELECTORS[1:]] + \
          [("S5 Learned selector", b) for b in ["S2 Best fixed subset", "S3 Confidence rule", "S4 Probability average", "R1 M1 text-only model"]]
rows = [{"comparison": f"{SEL_TAG[a]} − {SEL_TAG[b]}", "a": a, "b": b, **grouped_boot(test_pred[a], test_pred[b])} for a, b in PAIRS_B]
cmpB = pd.DataFrame(rows)
cmpB["video_p_holm"] = holm(cmpB["video_p"]); cmpB["seed+video_p_holm"] = holm(cmpB["seed+video_p"])
cmpB.to_csv(f"{OUT_DIR}/expB_comparisons_bootstrap.csv", index=False)
cmpB.drop(columns=["a", "b"]).round(4)

 seed  chosen_alpha  chosen_switch_fraction best_fixed_subset  spearman_pred_vs_true_delta  AUROC_detect_AV_hurts(true_delta<0)  AUROC_detect_fixable_clips  AUROC_confidence_rule_fixable  clips_switched  switched_and_fixed  switched_and_broken  fixable_clips_total  breakable_clips_total  accuracy_ceiling_gain_pp
   42         100.0                    0.30                 T                       0.1909                               0.5911                      0.6359                         0.4985            1184                  28                   30                   54                    116                    1.1590
   43           0.1                    0.30             T+A+V                       0.1363                               0.5874                      0.4017                         0.4718            1425                  35                   14                  102                    163                    2.1893
   44         100.0                    0.02               

,comparison,mean_diff_pp,per_seed_pp,sd_over_seeds_pp,seeds_positive,video_CI95_pp,video_p,seed+video_CI95_pp,seed+video_p,video_p_holm,seed+video_p_holm
0,S1 − S0,-0.7783,"-1.17, -0.51, -0.65",0.3509,0/3,"[-1.29, -0.28]",0.0026,"[-1.43, -0.09]",0.0280,0.0208,0.2520
1,S2 − S0,-0.5873,"-1.17, +0.00, -0.59",0.5875,0/3,"[-0.91, -0.27]",0.0002,"[-1.32, +0.00]",0.0998,0.0020,0.6033
2,S3 − S0,-0.3360,"-0.67, -0.07, -0.27",0.3054,0/3,"[-0.66, -0.02]",0.0370,"[-0.82, +0.14]",0.1596,0.1850,0.7979
3,S4 − S0,-0.2594,"-0.59, +0.05, -0.25",0.3207,1/3,"[-0.56, +0.04]",0.0868,"[-0.74, +0.21]",0.2688,0.3472,1.0000
4,S5 − S0,0.0667,"-0.21, +0.40, +0.00",0.3084,2/3,"[-0.10, +0.23]",0.4128,"[-0.30, +0.43]",0.6941,0.9509,1.0000
5,R1 − S0,-0.2581,"-0.84, +0.73, -0.66",0.8636,1/3,"[-0.91, +0.40]",0.4500,"[-1.36, +0.90]",0.6293,0.9509,1.0000
6,S5 − S2,0.6540,"+0.97, +0.40, +0.59",0.2889,3/3,"[+0.35, +0.97]",0.0002,"[+0.20, +1.16]",0.0074,0.0020,0.0740
7,S5 − S3,0.4026,"+0.46, +0.47, +0.27",0.1130,3/3,"[+0.11, +0.70]",0.0074,"[+0.01, +0.76]",0.0454,0.0518,0.3632
8,S5 − S4,0.3261,"+0.38, +0.35, +0.25",0.0672,3/3,"[+0.03, +0.62]",0.0298,"[-0.05, +0.69]",0.0862,0.1788,0.6033
9,S5 − R1,0.3248,"+0.64, -0.33, +0.67",0.5691,2/3,"[-0.31, +0.96]",0.3170,"[-0.62, +1.27]",0.4854,0.9509,1.0000


### 7c. Inference cost of each selector (measured on this machine, test set)
Learned-selector cost = 5 M4 passes + feature building + ridge prediction. Training-time cost of fitting the ridge model is
not included (it is fitted once on validation and is negligible next to model training).

In [13]:
PASSES = {"S0 M4 (T+A+V)": 1, "S1 Text only": 1, "S2 Best fixed subset": 1, "S3 Confidence rule": 2,
          "S4 Probability average": 2, "S5 Learned selector": 5}
cost = pd.DataFrame([{"selector": k, "M4_forward_passes": n,
                      "seconds_for_test_set": n * SEC_PER_PASS + (np.median(ridge_seconds) if k.startswith("S5") else 0.0),
                      "relative_to_S0": (n * SEC_PER_PASS + (np.median(ridge_seconds) if k.startswith("S5") else 0.0)) / SEC_PER_PASS}
                     for k, n in PASSES.items()])
cost["ms_per_clip"] = 1000 * cost["seconds_for_test_set"] / len(y_test)
print(f"Ridge features + prediction, median over seeds: {np.median(ridge_seconds):.3f} s for the test set")
print("R1 (M1) is a separate, much smaller model and is not timed here.")
cost.to_csv(f"{OUT_DIR}/expB_inference_cost.csv", index=False)
cost.round(3)

Ridge features + prediction, median over seeds: 0.005 s for the test set
R1 (M1) is a separate, much smaller model and is not timed here.


,selector,M4_forward_passes,seconds_for_test_set,relative_to_S0,ms_per_clip
0,S0 M4 (T+A+V),1,1.363,1.000,0.292
1,S1 Text only,1,1.363,1.000,0.292
2,S2 Best fixed subset,1,1.363,1.000,0.292
3,S3 Confidence rule,2,2.725,2.000,0.585
4,S4 Probability average,2,2.725,2.000,0.585
5,S5 Learned selector,5,6.818,5.004,1.463


## 8. Files written (v1/v2/v3 folders untouched)

In [14]:
for root, _, files in os.walk(OUT_DIR):
    for fn in sorted(files):
        print("  ", os.path.relpath(os.path.join(root, fn), OUT_DIR))

   control_check.json
   expA_dedicated_vs_M4_subset.csv
   expA_unimodal_per_seed.csv
   expB_comparisons_bootstrap.csv
   expB_inference_cost.csv
   expB_predictor_diagnostics.csv
   expB_selectors_per_seed.csv
   expB_validation_cv.csv
   run_info.json
   test_ids.csv
   test_labels.npy
   test_scores.npy
   valid_ids.npy
   valid_labels.npy
   checkpoints/U0_seed42.pt
   checkpoints/U0_seed42_log.json
   checkpoints/U0_seed42_test_probs.npy
   checkpoints/U0_seed42_valid_probs.npy
   checkpoints/U1_seed42.pt
   checkpoints/U1_seed42_log.json
   checkpoints/U1_seed42_test_probs.npy
   checkpoints/U1_seed42_valid_probs.npy
   checkpoints/U1_seed43.pt
   checkpoints/U1_seed43_log.json
   checkpoints/U1_seed43_test_probs.npy
   checkpoints/U1_seed43_valid_probs.npy
   checkpoints/U1_seed44.pt
   checkpoints/U1_seed44_log.json
   checkpoints/U1_seed44_test_probs.npy
   checkpoints/U1_seed44_valid_probs.npy
   checkpoints/U2_seed42.pt
   checkpoints/U2_seed42_log.json
   checkpoints/U2_s